In [ ]:
# --- ICESat-2 download via earthaccess with AOI shapefile + multi-track filtering ---
# Install once (separate cell / terminal):
#   pip install earthaccess geopandas shapely pyproj fiona

import os, re, json
import geopandas as gpd
from shapely.ops import unary_union
from shapely.geometry import Polygon, MultiPolygon, LineString, Point
from shapely.validation import make_valid
from shapely.geometry.polygon import orient
import earthaccess as ea

# ===================== USER SETTINGS =====================
SHAPEFILE_PATH = r"path\to\your\aoi.shp" # AOI shapefile (must be in WGS84 lat/lon)
OUTPUT_DIR     = r"path\to\your\output\directory" # where to save downloaded files; subfolders will be created if SPLIT_BY_TRACK=True

SHORT_NAME     = "ATL06"      # e.g., ATL03 (photons), ATL06 (heights), ATL08, ATL12
VERSION        = "007"        # set None to accept any (ATL06 usually "007")
TIME_START     = "2019-01-01"
TIME_END       = "2025-12-31"
CLOUD_HOSTED   = True         # True: prefer AWS cloud-hosted; False: NSIDC Direct

# —— Choose tracks ——
# Use None to download ALL tracks that intersect your AOI/time, or a list like ["129","731"]
TRACKS         =  ["1173"]  #["137"]    # <- example; ints or strings OK; they’ll be zero-padded to 4 digits  # set to None to download all tracks
SPLIT_BY_TRACK = True              # save each track to its own subfolder under OUTPUT_DIR
# =========================================================

In [ ]:
# ---------------- Helpers ----------------
def to_single_polygon(g):
    """Coerce any geometry into a single valid Polygon for CMR."""
    if g is None or g.is_empty:
        raise ValueError("AOI geometry is empty.")
    if isinstance(g, Polygon):
        poly = g
    elif isinstance(g, MultiPolygon):
        poly = max(g.geoms, key=lambda x: x.area)
    elif isinstance(g, LineString):
        poly = g.buffer(0.001)  # ~100 m at equator
    elif isinstance(g, Point):
        poly = g.buffer(0.001)
    else:
        poly = g.convex_hull
    poly = make_valid(poly)
    if isinstance(poly, MultiPolygon):
        poly = max(poly.geoms, key=lambda x: x.area)
    if not isinstance(poly, Polygon) or poly.is_empty:
        raise ValueError("Could not produce a valid polygon from AOI.")
    return poly

def cmr_polygon_coords(poly, max_points=500):
    """
    Return exterior ring as list of [lon, lat] pairs for CMR, CLOSED and CCW.
    Handles 2D/3D coords and simplifies if too many vertices.
    """
    if not isinstance(poly, Polygon):
        raise ValueError("AOI is not a Polygon.")
    # Force CCW orientation
    poly = orient(poly, sign=1.0)
    # Simplify if too many vertices; enforce CCW again (simplify may flip)
    if len(poly.exterior.coords) > max_points:
        poly = orient(poly.simplify(0.0002, preserve_topology=True), sign=1.0)

    coords = []
    for c in poly.exterior.coords:
        x = float(c[0]); y = float(c[1])
        # normalize longitudes to [-180, 180] just in case
        if x > 180: x -= 360
        if x < -180: x += 360
        coords.append((x, y))

    if len(coords) < 4:
        raise ValueError(f"CMR polygon needs >= 4 coordinate pairs; got {len(coords)}.")
    # close ring (CMR requires last==first)
    if coords[0] != coords[-1]:
        coords.append(coords[0])
    if len(coords) < 5:
        raise ValueError("Closed polygon must contain at least 4 unique vertices (5 with closure).")
    return coords

def granule_name(g):
    """Prefer ProducerGranuleId or GranuleUR; fallback to any id-like string."""
    umm = getattr(g, "umm", None) or (g.get("umm", {}) if isinstance(g, dict) else {})
    if isinstance(umm, dict):
        dg  = umm.get("DataGranule", {}) or {}
        pid = dg.get("ProducerGranuleId")
        if pid: return str(pid)
        gur = umm.get("GranuleUR")
        if gur: return str(gur)
    return str(getattr(g, "granule_id", "") or getattr(g, "id", "") or g)

def extract_track_from_name(name: str):
    """
    Get the 4-digit RGT from an ICESat-2 granule name.
    Prefer underscore-splitting (robust), then regex fallback.
    Examples:
      ATL03_20190105212430_01290203_007_01.h5  -> 0129
      ATL06_20230714T101530_07310115_007_02.h5 -> 0731
    """
    s = str(name)
    parts = s.split("_")
    # prefer 8-digit (RGT+cycle) block: take first 4 digits as RGT
    for p in parts:
        if p.isdigit() and len(p) == 8:
            return p[:4]
    # then plain 4-digit block
    for p in parts:
        if p.isdigit() and len(p) == 4:
            return p
    # regex fallback (odd separators)
    m = re.search(r"[_.-](\d{8})[_.-]", s)
    if m: return m.group(1)[:4]
    m = re.search(r"[_.-](\d{4})[_.-]", s)
    if m: return m.group(1)
    return None

def extract_track(g):
    """Extract 4-digit RGT from a granule object using the filename."""
    rgt = extract_track_from_name(granule_name(g))
    return rgt.zfill(4) if rgt else None

def z4(x): 
    return str(x).zfill(4)


In [ ]:

# ---------------- Main ----------------
# 1) Read AOI and ensure WGS84
gdf = gpd.read_file(SHAPEFILE_PATH)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS. Define/reproject it, then retry.")
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

# 2) AOI -> polygon -> coords
geom = unary_union(gdf.geometry)
poly = to_single_polygon(geom)
coords = cmr_polygon_coords(poly)
print(f"AOI vertices: {len(coords)} | Bounds: {poly.bounds}")

# 3) Auth & search
ea.login(strategy="netrc")  # prompts if not configured

kwargs = {
    "short_name": SHORT_NAME,
    "temporal": (TIME_START, TIME_END),
    "polygon": coords,
    "cloud_hosted": CLOUD_HOSTED
}
if VERSION is not None:
    kwargs["version"] = VERSION

# 4) Search by polygon; fallback to bbox if needed
try:
    results = ea.search_data(**kwargs)
except Exception as e:
    print(f"Polygon search raised: {e}\nFalling back to bounding box…")
    minx, miny, maxx, maxy = poly.bounds
    kwargs_bb = {
        "short_name": SHORT_NAME,
        "temporal": (TIME_START, TIME_END),
        "bounding_box": (minx, miny, maxx, maxy),
        "cloud_hosted": CLOUD_HOSTED
    }
    if VERSION is not None:
        kwargs_bb["version"] = VERSION
    results = ea.search_data(**kwargs_bb)

print(f"Found {len(results)} {SHORT_NAME} granule(s) before track filtering.")
for g in results[:5]:
    print("  •", granule_name(g))

# 5) Multi-track filter & download
requested = None if (not TRACKS) else {z4(t) for t in TRACKS}
print("Requested tracks:", sorted(requested) if requested else "(all)")

# Parse RGT for each result
by_track = {}
unknown = []
for g in results:
    rgt = extract_track(g)  # returns 4-digit string or None
    if rgt is None:
        unknown.append(g)
    else:
        by_track.setdefault(rgt, []).append(g)

print(f"Parsed tracks found: {sorted(by_track.keys())}  |  unknown={len(unknown)}")

# Decide batches to download
if not requested:
    batches = {"ALL": results}  # no filtering
else:
    batches = {rgt: by_track.get(rgt, []) for rgt in requested}
    missing = [r for r in requested if not batches[r]]
    if missing:
        print("No granules for tracks:", sorted(missing))

# print how many tracks to download
print(f"Downloading {len(batches)} track(s):", sorted(batches.keys()))

# print number of files per track
for rgt, granules in batches.items():
    print(f"  Track {rgt}: {len(granules)} file(s)")

    if dl:
        total_bytes = sum(os.path.getsize(p) for p in dl)
        print(f"Total size: {total_bytes / (1024**2):.2f} MB ({total_bytes / (1024**3):.2f} GB)")
    else:
        print("No files downloaded.")


In [ ]:
# 6) Download
os.makedirs(OUTPUT_DIR, exist_ok=True)
total = 0
for rgt, granules in batches.items():
    if not granules:
        continue
    subdir = os.path.join(OUTPUT_DIR, f"{rgt}") if SPLIT_BY_TRACK and rgt != "ALL" else OUTPUT_DIR
    os.makedirs(subdir, exist_ok=True)
    try:
        dl = ea.download(granules, local_path=subdir, keep_structure=False)
    except TypeError:
        dl = ea.download(granules, local_path=subdir)
    print(f"Downloaded {len(dl)} file(s) for track {rgt} → {subdir}")
    total += len(dl)

if total == 0:
    print("No granules to download. Consider adjusting time window, version, AOI, or requested tracks.")

# 7) Save query metadata
meta = {
    "short_name": SHORT_NAME,
    "version": VERSION,
    "temporal": {"start": TIME_START, "end": TIME_END},
    "polygon_coords_len": len(coords),
    "output_dir": OUTPUT_DIR,
    "cloud_hosted": CLOUD_HOSTED,
    "tracks_requested": (sorted(requested) if requested else "(all)"),
    "tracks_found": sorted(by_track.keys())
}
with open(os.path.join(OUTPUT_DIR, f"{SHORT_NAME}_download_query.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("Saved query metadata.")